# UNCP 2 | Model and executed results

کد مدل و یک بررسی اجرایی در همین دفترچه اجرا شده‌اند. اعداد سه seed از فایل‌های نتیجهٔ آزمایش کامل که پیش‌تر اجرا شده‌اند، در سلول آخر ثبت و با میانگین همان فایل‌ها کنترل می‌شوند. این دفترچه آموزش کامل سه seed را دوباره اجرا نمی‌کند؛ برای بازآموزی، داده، کد benchmark و تنظیمات همان پروژه لازم است.

In [1]:
import torch
from torch import nn
import torch.nn.functional as F
import math
import torch

def sinusoidal_positions(max_context, dimension):
    pe = torch.zeros(max_context, dimension)
    positions = torch.arange(max_context)[:, None]
    div = torch.exp(torch.arange(0, dimension, 2) * (-math.log(10000.0) / dimension))
    pe[:, 0::2] = torch.sin(positions * div)
    pe[:, 1::2] = torch.cos(positions * div)
    return pe

import torch
from torch import nn
import torch.nn.functional as F



class UNCP2NextClick(nn.Module):
    """UNCP 2: graph-aware GRU--Transformer fusion for next-click prediction.

    The graph is built from training-only click transitions. A symmetric
    normalized adjacency propagates element embeddings before the recurrent
    and attention branches, while a pooled graph state provides a third
    representation to the learned fusion gate.
    """

    def __init__(self, vocab_size, n_classes, config):
        super().__init__()
        dim, hidden = config["embedding_dim"], config["hidden_dim"]
        self.embedding = nn.Embedding(vocab_size, dim, padding_idx=0)
        adjacency = config.get("graph_adj")
        if adjacency is None:
            adjacency = torch.eye(vocab_size)
        self.register_buffer("graph_adj", adjacency.float())
        self.graph_self = nn.Linear(dim, dim)
        self.graph_neigh = nn.Linear(dim, dim, bias=False)
        self.graph_norm = nn.LayerNorm(dim)
        self.gru = nn.GRU(dim, hidden, batch_first=True)
        layer = nn.TransformerEncoderLayer(
            d_model=dim, nhead=config["attention_heads"],
            dim_feedforward=config["transformer_feedforward"],
            dropout=config["dropout"], batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(
            layer, config["transformer_layers"], enable_nested_tensor=False)
        self.register_buffer("positional", sinusoidal_positions(config["max_context"], dim))
        self.transform_projection = nn.Linear(dim, hidden)
        self.graph_projection = nn.Linear(dim, hidden)
        self.gate = nn.Linear(3 * hidden, hidden)
        self.norm = nn.LayerNorm(hidden)
        self.dropout = nn.Dropout(config["dropout"])
        self.output = nn.Linear(hidden, n_classes)

    def forward(self, x, lengths):
        x = x[:, :int(lengths.max())]
        batch = torch.arange(len(x), device=x.device)
        node_embeddings = self.embedding.weight
        propagated = torch.matmul(self.graph_adj, node_embeddings)
        graph_nodes = self.graph_norm(F.gelu(self.graph_self(node_embeddings) + self.graph_neigh(propagated)))
        embedded = self.embedding(x)
        graph_tokens = graph_nodes[x]
        token_features = embedded + graph_tokens
        recurrent, _ = self.gru(token_features)
        recurrent = recurrent[batch, lengths - 1]
        attended = self.transformer(
            token_features + self.positional[:x.shape[1]], src_key_padding_mask=x.eq(0))
        attended = self.transform_projection(attended[batch, lengths - 1])
        observed = (~x.eq(0)).float().unsqueeze(-1)
        graph_global = (graph_tokens * observed).sum(1) / observed.sum(1).clamp_min(1.0)
        graph_global = self.graph_projection(graph_global)
        states = torch.cat([recurrent, attended, graph_global], dim=1)
        gate = torch.sigmoid(self.gate(states))
        fused = self.norm(gate * recurrent + (1.0 - gate) * 0.5 * (attended + graph_global))
        return self.output(self.dropout(fused))


In [2]:
cfg=dict(embedding_dim=32,hidden_dim=64,attention_heads=4,transformer_feedforward=64,dropout=.2,transformer_layers=2,max_context=32,graph_adj=torch.eye(10))
model=UNCP2NextClick(10,5,cfg)
x=torch.tensor([[2,2,3,0],[4,5,6,7]])
lengths=torch.tensor([3,4])
out=model(x,lengths)
assert out.shape==(2,5) and torch.isfinite(out).all()
print('UNCP 2 forward pass OK:',tuple(out.shape))

UNCP 2 forward pass OK: (2, 5)


## Recorded experiment results

The following cell executes a consistency check over the recorded three-seed result rows and displays them. It does not retrain the model.

In [3]:
import json, numpy as np
record = json.loads("{\"summary\": {\"protocol\": \"grouped_user\", \"model\": \"uncp2\", \"seeds\": \"3\", \"test_examples\": \"10833\", \"test_users\": \"11\", \"top1\": \"0.646173728422413\", \"top3\": \"0.8357795624480753\", \"macro_target_top1\": \"0.5566763752880068\", \"macro_target_top3\": \"0.710277248842001\", \"macro_user_top1\": \"0.6547269754253718\", \"macro_user_top3\": \"0.841734839463901\", \"cluster_ci95_top1_low\": \"0.5965644564681961\", \"cluster_ci95_top1_high\": \"0.6865277802703998\", \"cluster_ci95_top3_low\": \"0.8044536926550897\", \"cluster_ci95_top3_high\": \"0.8595455960087095\", \"top1_seed_sd\": \"0.010989606710154688\", \"top3_seed_sd\": \"0.00042302000322679944\", \"macro_target_top1_seed_sd\": \"0.01894868669322078\", \"macro_target_top3_seed_sd\": \"0.004574994680479748\", \"macro_user_top1_seed_sd\": \"0.009704568969272733\", \"macro_user_top3_seed_sd\": \"0.0011327961284611167\"}, \"runs\": [{\"protocol\": \"grouped_user\", \"model\": \"uncp2\", \"seed\": \"11\", \"test_examples\": \"10833\", \"test_users\": \"11\", \"top1\": \"0.6569740607403305\", \"top3\": \"0.8356872519154436\", \"macro_target_top1\": \"0.5701158946043721\", \"macro_target_top3\": \"0.7122130350491778\", \"macro_user_top1\": \"0.6636911729202241\", \"macro_user_top3\": \"0.8418293052482178\", \"best_epoch\": \"29\", \"epochs_run\": \"30\"}, {\"protocol\": \"grouped_user\", \"model\": \"uncp2\", \"seed\": \"23\", \"test_examples\": \"10833\", \"test_users\": \"11\", \"top1\": \"0.6350041539739685\", \"top3\": \"0.8354103203175482\", \"macro_target_top1\": \"0.5350037825268091\", \"macro_target_top3\": \"0.7050525817079413\", \"macro_user_top1\": \"0.6444215587805323\", \"macro_user_top3\": \"0.8405577684283118\", \"best_epoch\": \"24\", \"epochs_run\": \"29\"}, {\"protocol\": \"grouped_user\", \"model\": \"uncp2\", \"seed\": \"37\", \"test_examples\": \"10833\", \"test_users\": \"11\", \"top1\": \"0.6465429705529401\", \"top3\": \"0.8362411151112342\", \"macro_target_top1\": \"0.5649094487328397\", \"macro_target_top3\": \"0.7135661297688842\", \"macro_user_top1\": \"0.6560681945753589\", \"macro_user_top3\": \"0.8428174447151737\", \"best_epoch\": \"28\", \"epochs_run\": \"30\"}], \"behavior\": null}")
summary, runs = record['summary'], record['runs']
top1 = float(summary.get('test_mean_top1', summary.get('top1')))
top3 = float(summary.get('test_mean_top3', summary.get('top3')))
seed_top1 = np.array([float(row['top1']) for row in runs])
seed_top3 = np.array([float(row['top3']) for row in runs])
assert np.isclose(seed_top1.mean(), top1, atol=1e-9)
assert np.isclose(seed_top3.mean(), top3, atol=1e-9)
print('Recorded grouped-user experiment: 10,833 test targets, 11 held-out users')
print(f'UNCP 2  Top-1: {100*top1:.2f}%  Top-3: {100*top3:.2f}%')
print('Seeds 11, 23, 37 Top-3:', [round(100*x,2) for x in seed_top3])
if record['behavior']:
    for group in record['behavior']:
        print(group['group'], 'n=',group['n'],'UNCP6 Top-3=',round(100*group['uncp6_top3'],2),'UNCP5 Top-3=',round(100*group['uncp5_top3'],2))

Recorded grouped-user experiment: 10,833 test targets, 11 held-out users
UNCP 2  Top-1: 64.62%  Top-3: 83.58%
Seeds 11, 23, 37 Top-3: [np.float64(83.57), np.float64(83.54), np.float64(83.62)]
